# 02 — Image Processing

Preprocessing pipeline:

```
CDL (USA-wide, EPSG:5070, 30 m)
  └─ Step 1: Reproject EPSG:5070 → EPSG:4326  +  clip to study area  +  resample to S2 res
  └─ Step 2: Filter CDL classes (keep target crops only)

S2 images (EPSG:4326, ~10 m, study area)
  └─ Step 3: Assign NoData  (invalid / negative / NaN → -9999, cast to float32)

Verify: CDL and S2 share the same CRS, bounds, and pixel grid
```

> **Why CDL first?**  The CDL raster covers the entire USA.  
> Reprojecting + clipping it to the study area *before* any S2 work keeps all subsequent operations cheap.

## Configuration

In [ ]:
import sys, os
sys.path.append("../")

# ── S2 sample files (3 dates, chronological order) ────────────────────────────
# Using local Google Drive mount directly.
# To download via gdown instead, see the cell below.
S2_DIR = "/Users/dikaizm/Library/CloudStorage/GoogleDrive-dikamaah@gmail.com/My Drive/S2_Annual_15d_sacramento_3"
S2_SAMPLES = [
    os.path.join(S2_DIR, "S2H_2022_2022_01_01.tif"),
    os.path.join(S2_DIR, "S2H_2022_2022_01_16.tif"),
    os.path.join(S2_DIR, "S2H_2022_2022_01_31.tif"),
]

# ── CDL label (USA-wide) ──────────────────────────────────────────────────────
CDL_RAW = "/Volumes/T7/research-crop-mapping-geoai/data/cdl/2022_30m_cdls/2022_30m_cdls.tif"

# ── Output directory ──────────────────────────────────────────────────────────
OUT_DIR     = "../data/processed"
OUT_CDL_DIR = os.path.join(OUT_DIR, "cdl")
OUT_S2_DIR  = os.path.join(OUT_DIR, "s2")
os.makedirs(OUT_CDL_DIR, exist_ok=True)
os.makedirs(OUT_S2_DIR,  exist_ok=True)

CDL_REPROJECTED = os.path.join(OUT_CDL_DIR, "cdl_2022_study_area.tif")            # after step 1
CDL_FILTERED    = os.path.join(OUT_CDL_DIR, "cdl_2022_study_area_filtered.tif")   # after step 2

# ── Parameters ────────────────────────────────────────────────────────────────
S2_NODATA_VALUE  = -9999.0
CDL_NODATA_VALUE = 0       # 0 = background / no-data in CDL

# CDL class IDs to keep — 11 classes, selected by ≥1% area threshold
# Covers 93.3% of all cropland and 78.2% of study area.
# Class imbalance ratio (max:min) = 22.6:1, manageable with weighted loss.
# Scientific rationale: see documents/reports/report_20260302.md §11
KEEP_CLASSES = [3, 6, 24, 36, 37, 54, 61, 69, 75, 76, 220]
#   3=Rice, 6=Sunflower, 24=Winter Wheat, 36=Alfalfa, 37=Other Hay,
#   54=Tomatoes, 61=Fallow, 69=Grapes, 75=Almonds, 76=Walnuts, 220=Plums

# Extended 13-class experiment (adds Safflower + Pistachios):
# KEEP_CLASSES = [3, 6, 24, 33, 36, 37, 54, 61, 69, 75, 76, 204, 220]

print("S2 samples:")
for f in S2_SAMPLES:
    exists = os.path.exists(f)
    print(f"  {'✅' if exists else '❌'} {os.path.basename(f)}")
print(f"\nCDL: {'✅' if os.path.exists(CDL_RAW) else '❌'} {CDL_RAW}")
print(f"\nKEEP_CLASSES ({len(KEEP_CLASSES)}): {KEEP_CLASSES}")

### (Optional) Download S2 samples via gdown
Run this cell only if you need to download the files instead of reading from Google Drive mount.

In [ ]:
# Uncomment and run to download via gdown
# import gdown, re
#
# GDRIVE_LINKS = [
#     "https://drive.google.com/open?id=1iyWjuXqUWc6oK_jAh99_JkIc8mzwCpjf&usp=drive_fs",
#     "https://drive.google.com/open?id=1u7iXqd2cNWHkdJIRS2U52J3SC10FfZWy&usp=drive_fs",
#     "https://drive.google.com/open?id=1mFGEMKdzTvMB-AKPz8MQceWpWCz3E6fa&usp=drive_fs",
# ]
#
# def _extract_id(url):
#     m = re.search(r'[?&]id=([a-zA-Z0-9_-]+)', url)
#     return m.group(1) if m else None
#
# os.makedirs(OUT_S2_DIR, exist_ok=True)
# for url in GDRIVE_LINKS:
#     fid = _extract_id(url)
#     out = gdown.download(id=fid, output=OUT_S2_DIR + "/", quiet=False, resume=True, fuzzy=True)
#     print(f"Downloaded: {out}")

---
## Inspect: CRS Mismatch
Before processing, confirm the coordinate system difference between CDL and S2.

In [ ]:
import rasterio

print("=" * 55)
with rasterio.open(S2_SAMPLES[0]) as src:
    s2_crs    = src.crs
    s2_bounds = src.bounds
    s2_res    = src.res
    s2_shape  = (src.width, src.height)
    s2_bands  = src.count
    print(f"S2  ({os.path.basename(S2_SAMPLES[0])})")
    print(f"  CRS       : {s2_crs}")
    print(f"  Bounds    : {s2_bounds}")
    print(f"  Resolution: {s2_res}")
    print(f"  Size      : {s2_shape[0]} x {s2_shape[1]}  |  {s2_bands} bands")

print()
with rasterio.open(CDL_RAW) as src:
    cdl_crs    = src.crs
    cdl_bounds = src.bounds
    cdl_res    = src.res
    cdl_shape  = (src.width, src.height)
    print(f"CDL ({os.path.basename(CDL_RAW)})")
    print(f"  CRS       : {cdl_crs}")
    print(f"  Bounds    : {cdl_bounds}")
    print(f"  Resolution: {cdl_res} m")
    print(f"  Size      : {cdl_shape[0]} x {cdl_shape[1]}")

print()
print(f"CRS match?  {'✅ Yes' if s2_crs == cdl_crs else '❌ No — reprojection required'}")
print("=" * 55)

---
## Step 1 — Reproject, Clip & Resample CDL

Three operations in **one pass** using the S2 image as the reference grid:

| | CDL before | CDL after |
|---|---|---|
| CRS | EPSG:5070 (Conus Albers) | EPSG:4326 (WGS84) |
| Coverage | Entire USA | Study area only |
| Resolution | 30 m | ~10 m (matches S2) |

Using `Resampling.nearest` to preserve integer class labels.

In [ ]:
import numpy as np
from rasterio.warp import reproject, Resampling, calculate_default_transform

# ── Reference grid from S2 ────────────────────────────────────────────────────
with rasterio.open(S2_SAMPLES[0]) as s2_ref:
    target_crs       = s2_ref.crs
    target_transform = s2_ref.transform
    target_width     = s2_ref.width
    target_height    = s2_ref.height

# ── Reproject CDL onto S2 grid (reproject + clip + resample in one step) ──────
with rasterio.open(CDL_RAW) as cdl_src:
    dst_data = np.zeros((1, target_height, target_width), dtype=np.uint8)

    reproject(
        source=rasterio.band(cdl_src, 1),
        destination=dst_data,
        src_transform=cdl_src.transform,
        src_crs=cdl_src.crs,
        dst_transform=target_transform,
        dst_crs=target_crs,
        resampling=Resampling.nearest,
        dst_nodata=CDL_NODATA_VALUE,
    )

    profile = cdl_src.profile.copy()
    profile.update(
        crs=target_crs,
        transform=target_transform,
        width=target_width,
        height=target_height,
        nodata=CDL_NODATA_VALUE,
        dtype="uint8",
        count=1,
        compress="lzw",
    )

    with rasterio.open(CDL_REPROJECTED, "w", **profile) as dst:
        dst.write(dst_data)

print(f"✅ CDL reprojected → {CDL_REPROJECTED}")

with rasterio.open(CDL_REPROJECTED) as src:
    print(f"   CRS        : {src.crs}")
    print(f"   Size       : {src.width} x {src.height}")
    print(f"   Resolution : {src.res}")
    print(f"   Bounds     : {src.bounds}")

---
## Step 2 — Filter CDL Classes
Keep only the target crop classes; all other pixels are set to 0 (background).

In [ ]:
from utils import label as label_utils
from utils.constants import USDA_CDL_NAMES

label_utils.label_filtering(
    in_path=CDL_REPROJECTED,
    out_path=CDL_FILTERED,
    keep_classes=KEEP_CLASSES,
)

# Verify class distribution
with rasterio.open(CDL_FILTERED) as src:
    data = src.read(1)
    unique, counts = np.unique(data, return_counts=True)
    total = data.size
    print(f"CDL filtered → {CDL_FILTERED}")
    print(f"\n{'Class':>6}  {'Name':<30}  {'Pixels':>12}  {'%':>6}")
    print("-" * 60)
    for cls, cnt in zip(unique, counts):
        name = USDA_CDL_NAMES.get(int(cls), "background" if cls == 0 else "unknown")
        print(f"  {cls:>4}  {name:<30}  {cnt:>12,}  {cnt/total*100:>5.2f}%")

---
## Step 3 — Assign NoData: S2 Images
Loop over all 3 sample images.  Replace negative, NaN, and Inf values with the nodata sentinel and cast to float32.

In [ ]:
S2_PROCESSED = []   # collect output paths

for src_path in S2_SAMPLES:
    fname    = os.path.basename(src_path)
    out_path = os.path.join(OUT_S2_DIR, fname.replace(".tif", "_processed.tif"))

    with rasterio.open(src_path) as src:
        profile = src.profile.copy()
        profile.update(dtype="float32", nodata=S2_NODATA_VALUE, compress="lzw")
        data = src.read().astype(np.float32)

    invalid = (data < 0) | np.isnan(data) | np.isinf(data)
    data[invalid] = S2_NODATA_VALUE

    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(data)

    S2_PROCESSED.append(out_path)
    print(f"✅ {fname}  →  invalid pixels: {invalid.sum():,}")

print(f"\nProcessed {len(S2_PROCESSED)} S2 images → {OUT_S2_DIR}")

---
## Verify Alignment
CDL and each S2 image must share the same CRS, bounds, and pixel dimensions.

In [ ]:
with rasterio.open(CDL_FILTERED) as cdl:
    cdl_info = {
        "crs": cdl.crs,
        "bounds": cdl.bounds,
        "shape": (cdl.width, cdl.height),
        "res": cdl.res,
    }

print(f"{'File':<45}  {'CRS':<12}  {'W x H':<18}  {'Bounds match'}")
print("-" * 100)

for s2_path in S2_PROCESSED:
    with rasterio.open(s2_path) as src:
        crs_ok    = src.crs == cdl_info["crs"]
        shape_ok  = (src.width, src.height) == cdl_info["shape"]
        bounds_ok = src.bounds == cdl_info["bounds"]
        ok = "✅" if (crs_ok and shape_ok and bounds_ok) else "❌"
        print(f"{os.path.basename(s2_path):<45}  {str(src.crs):<12}  "
              f"{src.width} x {src.height:<10}  {ok}")

print()
print(f"CDL reference: CRS={cdl_info['crs']}, size={cdl_info['shape'][0]} x {cdl_info['shape'][1]}")